# VITS on the same Sinhala data, for a comparison that means something

A second architecture on **the same clips, the same held-out 80, the same ASCII text**, so
the difference between the two systems is the architecture and not the dataset.

**Set Accelerator to GPU T4 x2 before running.** A CLI push resets it and a Kaggle P100
cannot run this PyTorch at all; the preflight cell stops in seconds if you get one.

### Why this fine-tunes instead of training from scratch

VITS from scratch is a ~1M-step proposition. On 4.69 h and a T4 you would get maybe 150k
steps across several sessions and it would sound like it &mdash; a strawman, not a baseline.

This restores a pretrained English VITS instead, and **the reason that works is the same
trick the XTTS result rests on**: the text is already transliterated to ASCII, so an English
phoneme front-end can read it. Nothing is randomly initialised and the task is "learn an
accent" again. The transliteration buys transfer for both architectures, which is a better
story than either result alone.

### One model per voice

Piper ships one ONNX voice per speaker, and single-speaker &rarr; single-speaker fine-tuning
needs no surgery on a speaker-embedding table. Train `dinithi` first (4.69 h); `harini`
(2.11 h) is a second run.

### What this does not settle

Front-end is a second variable: XTTS reads the ASCII as BPE tokens, VITS reads it as espeak
phonemes. That is stated in the write-up rather than hidden. Controlling it properly means a
third run with a character front-end.

## 1. Install &mdash; restart the session after this cell

In [ ]:
# espeak-ng is not optional: the pretrained VITS is phoneme-based and coqui
# shells out to the espeak binary. Without it training fails at the first batch.
!apt-get -qq install -y espeak-ng > /dev/null 2>&1
!pip install -q "coqui-tts>=0.25.1" "coqui-tts-trainer>=0.2.0" soundfile librosa tensorboard
!espeak-ng --version

## 2. Preflight, code and paths

In [ ]:
import torch

name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
if name is None:
    raise RuntimeError("no GPU. Settings -> Accelerator -> GPU T4 x2.")
arch = "sm_%d%d" % torch.cuda.get_device_capability(0)
if arch not in torch.cuda.get_arch_list():
    raise RuntimeError("%s is %s; this torch has kernels for %s. "
                       "Settings -> Accelerator -> GPU T4 x2."
                       % (name, arch, " ".join(torch.cuda.get_arch_list())))
try:
    (torch.zeros(8, device="cuda") + 1).sum().item()
except Exception as exc:
    raise RuntimeError("%s cannot run this PyTorch: %s" % (name, exc))
print("GPU:", name, arch)

In [ ]:
import glob, os, shutil, subprocess

CODE = "/kaggle/temp/code"
if os.path.isdir(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/DSEgrp18/Dataset-creation-withEmotion.git",
                CODE], check=True)
SRC  = CODE + "/xtts_model_female"
VITS = CODE + "/vits_female"
print("code at", subprocess.run(["git", "-C", CODE, "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True).stdout.strip())


def sh(args, cwd=VITS):
    print("$", " ".join(args), flush=True)
    r = subprocess.run(args, cwd=cwd)
    if r.returncode != 0:
        raise RuntimeError("step failed with exit %d: %s" % (r.returncode, " ".join(args)))


hits = [p for p in glob.glob("/kaggle/input/**/*", recursive=True)
        if os.path.isdir(p) and "dinithi" in os.path.basename(p).lower()]
if not hits:
    raise RuntimeError("VoiceMakers dataset not attached.")
DATA = os.path.dirname(min(hits, key=lambda p: len(p.split("/"))))

# The XTTS export, attached so the scoring step can borrow its speaker encoder.
# Optional: training does not need it, only the comparison does.
xh = glob.glob("/kaggle/input/**/model_fp16.pth", recursive=True)
XTTS_EXPORT = os.path.dirname(min(xh, key=len)) if xh else None
xr = glob.glob("/kaggle/input/**/GPT_XTTS_si_female-*", recursive=True)
XTTS_RUN = max(xr, key=len) if xr else None

XDATASET = "/kaggle/temp/female_dataset"     # the XTTS dataset, rebuilt
VDATASET = "/kaggle/temp/vits_dataset"       # resampled, per-speaker
RUN      = "/kaggle/temp/vitsrun"
KEEP     = "/kaggle/working/vitsrun"
SYNTH    = "/kaggle/working/vits_synth"
SCORED   = "/kaggle/working/vits_scored"
print("data       :", DATA)
print("xtts export:", XTTS_EXPORT)
print("xtts run   :", XTTS_RUN)

## 3. The same dataset the XTTS run used

`prepare_voicemakers.py` with **the same seed and the same eval-per-speaker count**, so the
80 held-out clips are identical to the ones every XTTS number in RESULTS.md was measured on.
Change either and the comparison is void.

`prepare_vits.py` then resamples to 22.05 kHz and splits by speaker. It does not re-do the
filtering &mdash; it inherits it, which is the point.

In [ ]:
import urllib.request, os

VOCAB = "/kaggle/temp/vocab.json"
if not os.path.isfile(VOCAB):
    urllib.request.urlretrieve(
        "https://huggingface.co/coqui/XTTS-v2/resolve/main/vocab.json", VOCAB)

sh(["python", "prepare_voicemakers.py", "--src", DATA, "--out", XDATASET,
    "--speakers", "dinithi", "harini", "--vocab", VOCAB,
    "--eval-per-speaker", "40", "--seed", "1234"], cwd=SRC)

sh(["python", "prepare_vits.py", "--dataset", XDATASET, "--out", VDATASET,
    "--sample-rate", "22050"])

## 4. Smoke test &mdash; two minutes, and it catches nan

VITS fp16 is standard and roughly doubles throughput, which matters when the budget is one
Kaggle session. But the XTTS recipe in this repo disables fp16 because it drove that model to
nan in one step, so it is worth proving for this one rather than assuming. A few steps on a
tiny slice, then check the losses are finite.

In [ ]:
SPEAKER = "dinithi"          # 4.69 h -- the larger voice. harini is a second run.

import subprocess, sys
sys.path.insert(0, SRC)
import train_log

SMOKE = "/kaggle/working/vits_smoke.log"
cmd = ["python", "train_vits_female.py", "--dataset", VDATASET,
       "--speaker", SPEAKER, "--out", "/kaggle/temp/vits_smoke",
       "--smoke", "--batch-size", "4"]
print("$", " ".join(cmd), flush=True)
with open(SMOKE, "w") as fh:
    rc = subprocess.run(cmd, cwd=VITS, stdout=fh, stderr=subprocess.STDOUT).returncode

txt = open(SMOKE, encoding="utf-8", errors="replace").read()
print(train_log.strip_ansi(txt)[-2500:] if rc != 0 else
      "\n".join(train_log.interesting(txt, 10)))
if rc != 0:
    raise RuntimeError("smoke test exited %d -- full output in %s" % (rc, SMOKE))
print("\nsmoke OK")

In [ ]:
!rm -rf /kaggle/temp/vits_smoke
!df -h /kaggle/working /kaggle/temp | grep -v Filesystem

## 5. Train

Budgeted the same way the XTTS run was, so "at equal GPU budget" is a claim the numbers can
support. Backgrounded when interactive, blocking under Save &amp; Run All.

In [ ]:
import os, shutil, subprocess, time, json, glob

BATCH = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "Interactive").lower() == "batch"
BUDGET_H = 8.5                      # the same budget train_xtts_female.py was given
LOG = "/kaggle/working/vits_train.log"
FLOOR_GB = 5.0

def free_gb(path):
    path = os.path.abspath(path)
    while not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent == path:
            break
        path = parent
    return shutil.disk_usage(path).free / 1e9

resume = None
prev = sorted(glob.glob(RUN + "/training/vits_si_female_*"), key=os.path.getmtime)
if prev:
    resume = prev[-1]
    print("resuming:", resume)

cmd = ["python", "train_vits_female.py", "--dataset", VDATASET, "--speaker", SPEAKER,
       "--out", RUN, "--batch-size", "16", "--save-step", "2000"]
if resume:
    cmd += ["--continue-path", resume]
print("$", " ".join(cmd), flush=True)

open(LOG, "w").close()
with open(LOG, "a") as fh:
    proc = subprocess.Popen(cmd, cwd=VITS, stdout=fh, stderr=subprocess.STDOUT)

if not BATCH:
    print("pid", proc.pid, "-- training in the background; re-run the next cell to follow")
else:
    started, deadline = time.time(), time.time() + BUDGET_H * 3600
    last, reason = 0.0, "budget"
    while proc.poll() is None and time.time() < deadline:
        time.sleep(30)
        if free_gb(RUN) < FLOOR_GB:
            reason = "disk"
            print("disk guard: %.1f GB left -- stopping" % free_gb(RUN), flush=True)
            break
        if time.time() - last > 600:
            last = time.time()
            txt = open(LOG, encoding="utf-8", errors="replace").read()
            print("[%.2f h left] " % ((deadline - time.time()) / 3600)
                  + " | ".join(t[-140:] for t in train_log.interesting(txt, 2)), flush=True)
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=300)
        except subprocess.TimeoutExpired:
            proc.kill()
    else:
        reason = "epochs_done" if proc.returncode == 0 else "process_exit"

    runs = glob.glob(RUN + "/training/vits_si_female_*")
    if runs:
        latest = max(runs, key=os.path.getmtime)
        dest = KEEP + "/" + os.path.basename(latest)
        os.makedirs(dest, exist_ok=True)
        for n in ("config.json", "best_model.pth"):
            if os.path.isfile(latest + "/" + n):
                shutil.copy2(latest + "/" + n, dest + "/" + n)
                print("kept %s (%.2f GB)" % (n, os.path.getsize(latest + "/" + n) / 1e9))
    json.dump({"reason": reason, "wall_h": round((time.time() - started) / 3600, 2),
               "budget_h": BUDGET_H, "speaker": SPEAKER},
              open("/kaggle/working/vits_run_status.json", "w"), indent=1)
    print("\nstop reason:", reason)

In [ ]:
!grep -E "loss_|EPOCH|EVAL|BEST" /kaggle/working/vits_train.log | tail -n 25

## 6. Speak the held-out set

One `<clip_id>.wav` per held-out item, named so the XTTS evaluator can score it.

In [ ]:
import glob, os

runs = glob.glob(RUN + "/training/vits_si_female_*")
if not runs:
    raise RuntimeError("no VITS run directory -- training produced no checkpoint")
vrun = max(runs, key=os.path.getmtime)
print("run:", vrun)

sh(["python", "synthesize_vits.py", "--run", vrun, "--dataset", VDATASET,
    "--out", SYNTH, "--speaker", SPEAKER, "--n", "40"])

## 7. Score it with the code that scored XTTS

`evaluate_xtts.py --synth-dir` computes every metric over audio it did not make, so both
architectures are judged by **one implementation** &mdash; including SECS, via the same XTTS
speaker encoder. That is why the XTTS export has to be attached: the model is loaded for its
encoder, not to synthesise anything.

In [ ]:
import os, shutil

if not (XTTS_EXPORT and XTTS_RUN):
    print("XTTS export/run not attached -- add Notebook Output "
          "uom230429e/xtts-new-optimised to score this against XTTS.")
else:
    XBASE = "/kaggle/temp/xbase"
    os.makedirs(XBASE, exist_ok=True)
    for f in ("config.json", "vocab.json"):
        if not os.path.isfile(XBASE + "/" + f):
            shutil.copy2(XTTS_EXPORT + "/" + f, XBASE + "/" + f)

    sh(["python", "evaluate_xtts.py", "--run", XTTS_RUN, "--base", XBASE,
        "--dataset", XDATASET, "--synth-dir", SYNTH, "--out", SCORED,
        "--checkpoint", XTTS_EXPORT + "/model_fp16.pth",
        "--n", "40", "--utmos", "--label", "vits-" + SPEAKER], cwd=SRC)

    from IPython.display import Markdown, display
    display(Markdown(open(SCORED + "/report.md", encoding="utf-8").read()))

## 8. Listen, and keep what matters

Download `vits_synth/`, `vitsrun/` and `vits_scored/` before the session ends. Then run the
`harini` voice as a second session by changing `SPEAKER` in cell 4.

In [ ]:
from IPython.display import Audio, display
import glob, json, os

ref = json.load(open(XDATASET + "/eval_reference.json", encoding="utf-8"))
shown = 0
for it in ref:
    if it["speaker"] != SPEAKER:
        continue
    syn = SYNTH + "/" + it["clip_id"] + ".wav"
    if not os.path.isfile(syn):
        continue
    print("\n" + it["sinhala"])
    print("  ->", it["ascii"])
    print("  REAL:");  display(Audio(XDATASET + "/" + it["wav"]))
    print("  VITS:");  display(Audio(syn))
    shown += 1
    if shown >= 4:
        break

print("\nkeep: %s, %s, %s" % (SYNTH, KEEP, SCORED))